<a href="https://colab.research.google.com/github/nikhiljedhe/RAG_AI_DUCUMENT_CHATBOT/blob/main/RAG_AI_chatbot_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-groq \
    faiss-cpu==1.9.0 \
    sentence-transformers==3.3.1 \
    pypdf==5.1.0 \
    streamlit==1.40.2 \
    pyngrok==7.2.0 \
    python-dotenv==1.0.1

In [2]:
GROQ_API_KEY = "xyz"    # your ap key here console.groq.com
NGROK_TOKEN  = "xyz"  # your auth token here dashboard.ngrok.com

import os
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print('✅ Keys set!')

✅ Keys set!


In [3]:
import os
os.makedirs('rag_chatbot', exist_ok=True)

with open('rag_chatbot/document_loader.py', 'w') as f:
    f.write('''
import tempfile, os
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

def load_document(uploaded_file):
    suffix = ".pdf" if uploaded_file.name.lower().endswith(".pdf") else ".txt"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as f:
        f.write(uploaded_file.read())
        tmp_path = f.name
    try:
        loader = PyPDFLoader(tmp_path) if suffix == ".pdf" else TextLoader(tmp_path, encoding="utf-8")
        documents = loader.load()
    finally:
        os.unlink(tmp_path)
    return documents

def chunk_documents(documents, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\\n\\n", "\\n", ". ", " ", ""]
    )
    return splitter.split_documents(documents)
''')

with open('rag_chatbot/vector_store.py', 'w') as f:
    f.write('''
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

def get_embeddings():
    return HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )

def build_vector_store(chunks):
    return FAISS.from_documents(chunks, get_embeddings())

def get_retriever(vector_store, k=4):
    return vector_store.as_retriever(search_kwargs={"k": k})
''')

with open('rag_chatbot/prompts.py', 'w') as f:
    f.write('''
from langchain.prompts import PromptTemplate

TEMPLATE = """You are a precise document assistant.
Answer the question using ONLY the context below.
If the answer is not in the context, say: "I could not find this information in the document."

Context:
{context}

Question: {question}

Answer (be concise and specific):"""

QA_PROMPT = PromptTemplate(template=TEMPLATE, input_variables=["context", "question"])
''')

with open('rag_chatbot/rag_pipeline.py', 'w') as f:
    f.write('''
import os
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from prompts import QA_PROMPT

def build_qa_chain(retriever, model_name="llama-3.1-8b-instant"):
    llm = ChatGroq(
        model_name=model_name,
        temperature=0,
        max_tokens=600,
        api_key=os.getenv("GROQ_API_KEY"),
    )
    return RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        chain_type_kwargs={"prompt": QA_PROMPT},
        return_source_documents=True,
    )

def query(qa_chain, question):
    result = qa_chain.invoke({"query": question})
    return result["result"], result["source_documents"]

def format_sources(source_documents):
    lines, seen = [], set()
    for doc in source_documents:
        page = doc.metadata.get("page", "?")
        snippet = doc.page_content[:120].replace("\\n", " ").strip()
        key = (page, snippet[:40])
        if key not in seen:
            seen.add(key)
            lines.append(f"📄 Page {page}: *{snippet}…*")
    return "\\n".join(lines)
''')

with open('rag_chatbot/app.py', 'w') as f:
    f.write('''
import streamlit as st
import os
from document_loader import load_document, chunk_documents
from vector_store import build_vector_store, get_retriever
from rag_pipeline import build_qa_chain, query, format_sources

os.environ.setdefault("GROQ_API_KEY", os.getenv("GROQ_API_KEY", ""))

st.set_page_config(page_title="DocChat AI", page_icon="📄", layout="wide")

with st.sidebar:
    st.title("⚙️ Settings")
    model_choice = st.selectbox("Groq Model", [
        "llama-3.1-8b-instant", "llama-3.3-70b-versatile", "llama-4-scout-17b-16e-instruct", "llama-4-maverick-17b-128e-instruct",
         "qwen/qwen3-32b",
    ])
    chunk_size = st.slider("Chunk Size", 300, 2000, 1000, 100)
    top_k = st.slider("Top-K Chunks", 1, 8, 4)
    st.divider()
    st.markdown("**Free Stack:**")
    st.markdown("- 🤖 LLM: Groq (LLaMA 3)")
    st.markdown("- 🧠 Embeddings: HuggingFace")
    st.markdown("- 🗂️ Vector DB: FAISS")
    if st.button("🗑️ Clear Chat & Reset", use_container_width=True):
        for k in ["qa_chain", "messages", "doc_name", "num_chunks"]:
            st.session_state.pop(k, None)
        st.rerun()

st.title("📄 AI Document Q&A Chatbot")
st.caption("Upload a PDF or TXT and ask questions — 100% free, no OpenAI key needed.")

uploaded_file = st.file_uploader("Upload a document", type=["pdf", "txt"])

if uploaded_file and (
    "qa_chain" not in st.session_state
    or st.session_state.get("doc_name") != uploaded_file.name
    or st.session_state.get("model") != model_choice
):
    with st.status("⚙️ Processing document...", expanded=True) as status:
        st.write("📂 Loading...")
        docs = load_document(uploaded_file)
        st.write("✂️ Chunking...")
        chunks = chunk_documents(docs, chunk_size=chunk_size)
        st.write("🧠 Generating embeddings (first run downloads ~90MB model)...")
        vs = build_vector_store(chunks)
        st.write(f"🔗 Building RAG chain with {model_choice}...")
        retriever = get_retriever(vs, k=top_k)
        st.session_state.qa_chain = build_qa_chain(retriever, model_name=model_choice)
        st.session_state.messages = []
        st.session_state.doc_name = uploaded_file.name
        st.session_state.num_chunks = len(chunks)
        st.session_state.model = model_choice
        status.update(label="✅ Ready!", state="complete")
    st.success(f"Indexed **{len(chunks)} chunks** from *{uploaded_file.name}*")

if "qa_chain" in st.session_state:
    st.divider()
    c1, c2, c3 = st.columns(3)
    c1.metric("Document", st.session_state.doc_name)
    c2.metric("Chunks", st.session_state.num_chunks)
    c3.metric("Model", st.session_state.model)
    st.divider()

    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.write(msg["content"])
            if msg["role"] == "assistant" and msg.get("sources"):
                with st.expander("📚 Sources"):
                    st.markdown(msg["sources"])

    if prompt := st.chat_input("Ask a question about your document..."):
        with st.chat_message("user"):
            st.write(prompt)
        with st.chat_message("assistant"):
            with st.spinner("Thinking..."):
                answer, source_docs = query(st.session_state.qa_chain, prompt)
                sources_text = format_sources(source_docs)
            st.write(answer)
            if sources_text:
                with st.expander("📚 Sources"):
                    st.markdown(sources_text)
        st.session_state.messages.append({"role": "user", "content": prompt})
        st.session_state.messages.append({"role": "assistant", "content": answer, "sources": sources_text})
else:
    st.info("👆 Upload a PDF or TXT file above to get started.")
''')

print('✅ All files written!')
!ls rag_chatbot/

✅ All files written!
app.py		    prompts.py	 rag_pipeline.py
document_loader.py  __pycache__  vector_store.py


In [4]:
import subprocess, time, os, sys
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_TOKEN

os.system('pkill -f streamlit 2>/dev/null; pkill -f ngrok 2>/dev/null')
time.sleep(2)

proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'rag_chatbot/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    env={**os.environ, 'GROQ_API_KEY': GROQ_API_KEY},
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)
tunnel = ngrok.connect(8501)
print(f'👉 YOUR APP LINK: {tunnel.public_url}')

ERROR:pyngrok.process.ngrok:t=2026-03-14T06:25:27+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: xyz\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-03-14T06:25:27+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: xyz\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: xyz\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.

In [7]:
import os
print(os.environ.get('NGROK_TOKEN', 'NOT FOUND'))

NOT FOUND
